## Cell 0: Inflation adjustment using CPI

This cell loads the movie dataset and adjusts budget and revenue values for inflation using approximate CPI values. It creates a year column from the release date, applies an inflation correction to both budget and revenue, and saves the updated dataset as inflation_adjusted_movies.xlsx.

In [35]:
import pandas as pd

# Load your dataset
df = pd.read_excel(r"C:\Users\swami\Desktop\Sem 4\MLPR\project\unbiased_fast_dataset.xlsx")

# -----------------------------
# CPI DATA (approx US CPI values)
# -----------------------------
cpi_data = {
    1980: 82.4, 1985: 107.6, 1990: 130.7, 1995: 152.4,
    2000: 172.2, 2005: 195.3, 2010: 218.1, 2015: 237.0,
    2018: 251.1, 2019: 255.7, 2020: 258.8, 2021: 270.9,
    2022: 292.7, 2023: 305.0, 2024: 315.0, 2025: 325.0
}

CURRENT_YEAR = 2025
CURRENT_CPI = cpi_data[CURRENT_YEAR]

# -----------------------------
# FUNCTION TO GET CPI
# -----------------------------
def get_cpi(year):
    if year in cpi_data:
        return cpi_data[year]
    else:
        # nearest year fallback
        closest_year = min(cpi_data.keys(), key=lambda x: abs(x - year))
        return cpi_data[closest_year]

# -----------------------------
# APPLY INFLATION ADJUSTMENT
# -----------------------------
def adjust_value(value, year):
    if pd.isna(value) or value == 0:
        return value

    cpi_year = get_cpi(year)
    return value * (CURRENT_CPI / cpi_year)

# Extract year from release_date
df["year"] = pd.to_datetime(df["release_date"], errors='coerce').dt.year

# Adjust values
df["adjusted_budget"] = df.apply(
    lambda row: adjust_value(row["budget"], row["year"]), axis=1
)

df["adjusted_revenue"] = df.apply(
    lambda row: adjust_value(row["revenue"], row["year"]), axis=1
)

# -----------------------------
# SAVE FILE
# -----------------------------
df.to_excel("inflation_adjusted_movies.xlsx", index=False)

print("✅ Inflation-adjusted dataset saved!")

✅ Inflation-adjusted dataset saved!


## Cell 1: Build an Indian movie dataset from TMDB

This cell collects movie IDs from TMDB for selected Indian languages and years, then fetches full movie details for each ID. It extracts key fields such as title, release date, language, budget, revenue, director, actors, producers, popularity, and vote average, and saves the final dataset as unbiased_indian_movies.xlsx.

## Cell 2: Final TMDB movie collection pipeline

This cell expands the TMDB scraping process further by scanning more years, more languages, and more pages. It collects movie IDs, fetches movie details in parallel, removes duplicates, and saves a larger dataset called indian_movies_full_dataset.xlsx.

In [1]:
import requests
import pandas as pd
import time
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

API_KEY = "903b99053bde149fce928141e82afeb0"
BASE_URL = "https://api.themoviedb.org/3"

THREADS = 15
LANGUAGES = ["hi", "te", "ta", "ml", "kn", "bn", "mr", "pa", "gu"]
YEARS = range(2025, 2000, -1)

# --- SESSION WITH RETRIES ---
session = requests.Session()
retries = Retry(
    total=5,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504]
)
session.mount('https://', HTTPAdapter(max_retries=retries))


# -----------------------------
# FETCH MOVIE DETAILS
# -----------------------------
def get_movie_details(movie_id):
    url = f"{BASE_URL}/movie/{movie_id}?api_key={API_KEY}&append_to_response=credits"
    try:
        time.sleep(0.03)

        res = session.get(url, timeout=10)
        if res.status_code != 200:
            return None

        data = res.json()

        crew = data.get("credits", {}).get("crew", [])
        cast = data.get("credits", {}).get("cast", [])

        return {
            "movie_id": data.get("id"),
            "title": data.get("title"),
            "release_date": data.get("release_date"),
            "original_language": data.get("original_language"),
            "budget": data.get("budget"),
            "revenue": data.get("revenue"),
            "director": next((c["name"] for c in crew if c["job"] == "Director"), "Unknown"),
            "actors": ", ".join([c["name"] for c in cast[:5]]),
            "producers": ", ".join([c["name"] for c in crew if c["job"] == "Producer"][:2]),
            "popularity": data.get("popularity"),
            "vote_average": data.get("vote_average")
        }

    except:
        return None


# -----------------------------
# STEP 1: COLLECT MOVIE IDs
# -----------------------------
potential_ids = set()

print("📡 Scanning Indian Cinema...")

for year in YEARS:
    for lang in LANGUAGES:
        for page in range(1, 80):

            url = (f"{BASE_URL}/discover/movie?api_key={API_KEY}"
                   f"&with_original_language={lang}"
                   f"&primary_release_year={year}"
                   f"&sort_by=primary_release_date.desc"
                   f"&page={page}")

            try:
                time.sleep(0.08)

                res = session.get(url, timeout=10)
                resp = res.json()

                if 'results' not in resp:
                    continue

                results = resp.get('results', [])

                if len(results) == 0:
                    break

                for m in results:
                    potential_ids.add(m['id'])

            except:
                continue

    print(f"✔ Year {year} done → IDs so far: {len(potential_ids)}")

print(f"\n✅ Total IDs collected: {len(potential_ids)}")


# -----------------------------
# STEP 2: FETCH DETAILS (SAVE ALL)
# -----------------------------
final_movies = []
seen_ids = set()

id_list = list(potential_ids)
random.shuffle(id_list)

print("\n🚀 Fetching movie details...")

with ThreadPoolExecutor(max_workers=THREADS) as executor:
    futures = [executor.submit(get_movie_details, mid) for mid in id_list]

    for i, future in enumerate(as_completed(futures)):
        result = future.result()

        # ✅ SAVE ALL MOVIES (no filter)
        if result and result['movie_id'] not in seen_ids:
            final_movies.append(result)
            seen_ids.add(result['movie_id'])

        if (i + 1) % 200 == 0:
            print(f"📦 Processed {i+1} | Saved: {len(final_movies)}")


# -----------------------------
# SAVE
# -----------------------------
df = pd.DataFrame(final_movies)
df.to_excel("indian_movies_full_dataset.xlsx", index=False)

print(f"\n🎉 DONE!")
print(f"Total Movies Saved: {len(df)}")

📡 Scanning Indian Cinema...
✔ Year 2025 done → IDs so far: 1545
✔ Year 2024 done → IDs so far: 3099
✔ Year 2023 done → IDs so far: 4596
✔ Year 2022 done → IDs so far: 6027
✔ Year 2021 done → IDs so far: 7156
✔ Year 2020 done → IDs so far: 7955
✔ Year 2019 done → IDs so far: 9162
✔ Year 2018 done → IDs so far: 10267
✔ Year 2017 done → IDs so far: 11274
✔ Year 2016 done → IDs so far: 12098
✔ Year 2015 done → IDs so far: 12877
✔ Year 2014 done → IDs so far: 13568
✔ Year 2013 done → IDs so far: 14226
✔ Year 2012 done → IDs so far: 14761
✔ Year 2011 done → IDs so far: 15264
✔ Year 2010 done → IDs so far: 15790
✔ Year 2009 done → IDs so far: 16275
✔ Year 2008 done → IDs so far: 16712
✔ Year 2007 done → IDs so far: 17125
✔ Year 2006 done → IDs so far: 17533
✔ Year 2005 done → IDs so far: 17936
✔ Year 2004 done → IDs so far: 18307
✔ Year 2003 done → IDs so far: 18656
✔ Year 2002 done → IDs so far: 18988
✔ Year 2001 done → IDs so far: 19286

✅ Total IDs collected: 19286

🚀 Fetching movie detail

## Cell 3: Add Wikipedia links to movies

This cell finds the Wikipedia page for each movie by first getting the Wikidata ID from TMDB’s external IDs, then converting that into the corresponding English Wikipedia title. The resulting Wikipedia link is stored in a new wikipedia_link column and saved as movies_with_wikipedia.xlsx.

In [2]:
import requests
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor

# -----------------------------
# CONFIG
# -----------------------------
API_KEY = "903b99053bde149fce928141e82afeb0"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

session = requests.Session()

# -----------------------------
# FUNCTION: TMDB → WIKIPEDIA
# -----------------------------
def get_wikipedia_link(movie_id):
    try:
        # STEP 1: TMDB → Wikidata ID
        url = f"https://api.themoviedb.org/3/movie/{movie_id}/external_ids?api_key={API_KEY}"
        res = session.get(url, timeout=10)

        if res.status_code != 200:
            return None

        wikidata_id = res.json().get("wikidata_id")
        if not wikidata_id:
            return None

        # STEP 2: Wikidata → Wikipedia title
        wd_url = f"https://www.wikidata.org/wiki/Special:EntityData/{wikidata_id}.json"
        wd_res = requests.get(wd_url, headers=HEADERS, timeout=10)

        if wd_res.status_code != 200:
            return None

        entity = wd_res.json()["entities"][wikidata_id]

        title = entity.get("sitelinks", {}).get("enwiki", {}).get("title")
        if not title:
            return None

        # STEP 3: Build Wikipedia URL
        return f"https://en.wikipedia.org/wiki/{title.replace(' ', '_')}"

    except:
        return None


# -----------------------------
# LOAD YOUR DATASET
# -----------------------------
df = pd.read_excel(r"C:\Users\swami\Desktop\Sem 4\MLPR\project\indian_movies_full_dataset.xlsx")

print(f"Total movies: {len(df)}")

# -----------------------------
# MULTITHREADED PROCESS
# -----------------------------
from concurrent.futures import ThreadPoolExecutor, as_completed

print("🌐 Fetching Wikipedia links...")

wiki_links = [None] * len(df)

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {
        executor.submit(get_wikipedia_link, row["movie_id"]): i
        for i, row in df.iterrows()
    }

    for count, future in enumerate(as_completed(futures)):
        i = futures[future]

        try:
            wiki_links[i] = future.result()
        except:
            wiki_links[i] = None

        # 🔥 PRINT PROGRESS
        if count % 50 == 0:
            print(f"Processed {count} / {len(df)} movies")

# -----------------------------
# ADD COLUMN
# -----------------------------
df["wikipedia_link"] = wiki_links

# -----------------------------
# SAVE FILE
# -----------------------------
df.to_excel("movies_with_wikipedia.xlsx", index=False)

print("🎉 DONE!")
print("Saved as: movies_with_wikipedia.xlsx")

# -----------------------------
# QUICK CHECK
# -----------------------------
print("\nSample:")
print(df[["title", "wikipedia_link"]].head())

print("\nNon-null Wikipedia links:")
print(df["wikipedia_link"].notnull().sum())

Total movies: 19286
🌐 Fetching Wikipedia links...
Processed 0 / 19286 movies
Processed 50 / 19286 movies
Processed 100 / 19286 movies
Processed 150 / 19286 movies
Processed 200 / 19286 movies
Processed 250 / 19286 movies
Processed 300 / 19286 movies
Processed 350 / 19286 movies
Processed 400 / 19286 movies
Processed 450 / 19286 movies
Processed 500 / 19286 movies
Processed 550 / 19286 movies
Processed 600 / 19286 movies
Processed 650 / 19286 movies
Processed 700 / 19286 movies
Processed 750 / 19286 movies
Processed 800 / 19286 movies
Processed 850 / 19286 movies
Processed 900 / 19286 movies
Processed 950 / 19286 movies
Processed 1000 / 19286 movies
Processed 1050 / 19286 movies
Processed 1100 / 19286 movies
Processed 1150 / 19286 movies
Processed 1200 / 19286 movies
Processed 1250 / 19286 movies
Processed 1300 / 19286 movies
Processed 1350 / 19286 movies
Processed 1400 / 19286 movies
Processed 1450 / 19286 movies
Processed 1500 / 19286 movies
Processed 1550 / 19286 movies
Processed 160

## Cell 4: Improved Wikipedia financial extraction with fallback

This cell is a stronger version of Wikipedia scraping. It first tries to extract budget and revenue from the infobox, and if that fails, it searches the paragraph text on the page. It only uses Wikipedia when TMDB budget or revenue is missing, then saves the updated dataset.

In [1]:
import pandas as pd
import requests
import time
import re
from bs4 import BeautifulSoup

# -----------------------------
# LOAD DATASET
# -----------------------------
df = pd.read_excel("C:\\Users\\swami\\Desktop\\Sem 4\\MLPR\\project\\movies_with_wikipedia.xlsx")
df=df.head(100)

print(f"Total movies: {len(df)}")

# -----------------------------
# HEADERS (VERY IMPORTANT)
# -----------------------------
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

# -----------------------------
# CLEAN TEXT
# -----------------------------
def clean_text(text):
    if not text:
        return None
    text = re.sub(r"\[.*?\]", "", text)  # remove [1], [2]
    return text.strip()

# -----------------------------
# EXTRACT MONEY VALUE
# -----------------------------
def extract_money(text):
    if not text:
        return None

    text = text.lower()

    match = re.search(r"(₹|\$|us\$)?\s?\d+[\d,.]*\s?(crore|million|billion)?", text)

    if match:
        return match.group().strip()

    return None

# -----------------------------
# STEP 1: INFOBOX EXTRACTION
# -----------------------------
def extract_financials(url):
    try:
        if pd.isna(url) or not url:
            return None, None

        res = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(res.text, "html.parser")

        infobox = soup.find("table", {"class": "infobox"})
        if not infobox:
            return None, None

        budget = None
        revenue = None

        for row in infobox.find_all("tr"):
            header = row.find("th")
            value = row.find("td")

            if not header or not value:
                continue

            h = header.text.lower()
            v = clean_text(value.text)

            if not budget and any(k in h for k in ["budget", "cost"]):
                budget = extract_money(v)

            if not revenue and any(k in h for k in ["box office", "gross", "revenue", "collection"]):
                revenue = extract_money(v)

        return budget, revenue

    except:
        return None, None

# -----------------------------
# STEP 2: PARAGRAPH FALLBACK
# -----------------------------
def extract_from_paragraph(url):
    try:
        if pd.isna(url) or not url:
            return None, None

        res = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(res.text, "html.parser")

        paragraphs = soup.find_all("p")

        budget = None
        revenue = None

        for p in paragraphs:
            text = p.get_text().lower()

            if not budget and ("budget" in text or "cost" in text):
                budget = extract_money(text)

            if not revenue and any(k in text for k in ["gross", "box office", "collection"]):
                revenue = extract_money(text)

            if budget and revenue:
                break

        return budget, revenue

    except:
        return None, None

# -----------------------------
# MAIN EXTRACTION LOOP
# -----------------------------
wiki_budgets = []
wiki_revenues = []

print("\n💰 Extracting financial data...\n")

for i, row in df.iterrows():

    tmdb_budget = row["budget"]
    tmdb_revenue = row["revenue"]

    wiki_budget = None
    wiki_revenue = None

    # 🔥 ONLY USE WIKIPEDIA IF TMDB IS MISSING
    if tmdb_budget == 0 or pd.isna(tmdb_budget) or tmdb_revenue == 0 or pd.isna(tmdb_revenue):

        b, r = extract_financials(row["wikipedia_link"])

        # fallback if infobox fails
        if not b or not r:
            b2, r2 = extract_from_paragraph(row["wikipedia_link"])

            if not b:
                b = b2
            if not r:
                r = r2

        wiki_budget = b
        wiki_revenue = r

    wiki_budgets.append(wiki_budget)
    wiki_revenues.append(wiki_revenue)

    # Progress
    if i % 20 == 0:
        print(f"Processed {i}/{len(df)}")

    time.sleep(0.2)  # avoid blocking

# -----------------------------
# ADD WIKI COLUMNS
# -----------------------------
df["wiki_budget"] = wiki_budgets
df["wiki_revenue"] = wiki_revenues

# -----------------------------
# MERGE DATA (SMART WAY)
# -----------------------------
df["budget"] = df["budget"].replace(0, None)
df["revenue"] = df["revenue"].replace(0, None)

df["budget"] = df["budget"].combine_first(df["wiki_budget"])
df["revenue"] = df["revenue"].combine_first(df["wiki_revenue"])

# -----------------------------
# SAVE FINAL DATASET
# -----------------------------
df.to_excel("final_movies_with_financials.xlsx", index=False)

print("\n🎉 DONE!")
print("Saved as: final_movies_with_financials.xlsx")

# -----------------------------
# CHECK RESULTS
# -----------------------------
print("\n📊 FINAL STATS:")
print("Non-null budgets:", df["budget"].notnull().sum())
print("Non-null revenues:", df["revenue"].notnull().sum())

print("\nSample:")
print(df[["title", "budget", "revenue"]].head())

Total movies: 100

💰 Extracting financial data...

Processed 0/100
Processed 20/100
Processed 40/100
Processed 60/100
Processed 80/100

🎉 DONE!
Saved as: final_movies_with_financials.xlsx

📊 FINAL STATS:
Non-null budgets: 27
Non-null revenues: 34

Sample:
                         title      budget revenue
0  On Either Sides of the Pond        None    None
1            Onnum Onnum Moonu        None    None
2                      License        None    None
3                       U Turn  ₹2.5 crore    None
4              Irumbu Kuthirai        None    None


## Cell 5: Convert Wikipedia financial text into numeric values

This cell cleans budget and revenue strings such as “₹50 Crore” or “$10 million” and converts them into numeric values. It then scrapes Wikipedia finance data row by row, updates the dataset with numeric budget and revenue, and saves the file as updated_movie_data.xlsx.

In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import os

def clean_currency(value):
    """Converts strings like '$10 million' or '₹50 Crore' to numeric values."""
    if pd.isna(value) or value == '0' or value == 0:
        return 0
    
    # Remove commas and convert to lowercase
    value = str(value).lower().replace(',', '')
    
    # Extract numeric part (handles decimals like 10.5)
    numbers = re.findall(r"[-+]?\d*\.\d+|\d+", value)
    if not numbers:
        return 0
    
    num = float(numbers[0])
    
    # Scale based on Indian and International suffixes
    if 'crore' in value:
        num *= 10_000_000
    elif 'million' in value:
        num *= 1_000_000
    elif 'billion' in value:
        num *= 1_000_000_000
    elif 'lakh' in value:
        num *= 100_000
        
    return int(num)

def scrape_wiki_finance(url):
    """Scrapes Wikipedia infobox for budget and revenue data."""
    try:
        # User-Agent is necessary to avoid being blocked by Wikipedia
        header = {'User-Agent': 'MovieDataBot/3.0 (Personal Academic Project)'}
        response = requests.get(url, headers=header, timeout=10)
        
        if response.status_code != 200:
            return {'budget': 0, 'revenue': 0}

        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Look for the infobox table
        infobox = soup.select_one("table.infobox")
        data = {'budget': 0, 'revenue': 0}
        
        if not infobox:
            return data

        for row in infobox.find_all("tr"):
            th = row.find("th")
            td = row.find("td")
            
            if th and td:
                label = th.get_text(separator=" ", strip=True).lower()
                
                # Use separator to prevent mashing text in multi-line lists
                value_text = td.get_text(separator=" ", strip=True)
                # Remove citations like [12] or [Note 1]
                value_text = re.sub(r'\[.*?\]', '', value_text).lower()
                
                # Logic for Budget
                if any(x in label for x in ['budget', 'cost', 'investment']):
                    data['budget'] = clean_currency(value_text)
                
                # Logic for Revenue
                elif any(x in label for x in ['gross', 'box office', 'revenue', 'collection']):
                    data['revenue'] = clean_currency(value_text)
                    
        return data
    except Exception:
        return {'budget': 0, 'revenue': 0}

# --- Main Execution ---

file_path = r"C:\Users\swami\Desktop\Sem 4\MLPR\project\movies_with_wikipedia.xlsx"

if not os.path.exists(file_path):
    print(f"ERROR: File not found at {file_path}")
else:
    # 1. Load your dataset
    df = pd.read_excel(file_path) 

    # Ensure budget and revenue columns exist as numeric types
    if 'budget' not in df.columns: df['budget'] = 0
    if 'revenue' not in df.columns: df['revenue'] = 0

    print(f"Starting analysis on {len(df)} movies...\n" + "-"*50)

    for index, row in df.iterrows():
        wiki_link = row.get('wikipedia_link')
        movie_title = row.get('title', 'Unknown')
        
        # Check if link exists
        if pd.isna(wiki_link) or str(wiki_link).strip() == "" or "nan" in str(wiki_link).lower():
            print(f"[{index}] SKIPPED: '{movie_title}' (No Wikipedia link)")
            continue
        
        # Scrape data using the corrected function name
        finance_data = scrape_wiki_finance(wiki_link)
        
        # Update dataframe
        df.at[index, 'budget'] = finance_data['budget']
        df.at[index, 'revenue'] = finance_data['revenue']
        
        if finance_data['budget'] > 0 or finance_data['revenue'] > 0:
            status = "FOUND"
        else:
            status = "EMPTY INFOBOX"
            
        print(f"[{index}] {status}: '{movie_title}' -> Budget: {finance_data['budget']}, Revenue: {finance_data['revenue']}")

    # --- Final Statistics ---

    non_zero_budget = (df['budget'] > 0).sum()
    non_zero_revenue = (df['revenue'] > 0).sum()

    print("-" * 50)
    print(f"TOTAL NON-ZERO BUDGET ROWS: {non_zero_budget}")
    print(f"TOTAL NON-ZERO REVENUE ROWS: {non_zero_revenue}")

    # Save the updated file to the same directory
    output_path = os.path.join(os.path.dirname(file_path), 'updated_movie_data.xlsx')
    df.to_excel(output_path, index=False)
    print(f"\nSuccess! Updated data saved to: {output_path}")

Starting analysis on 19286 movies...
--------------------------------------------------
[0] SKIPPED: 'Hawala' (No Wikipedia link)
[1] SKIPPED: 'Sethaan Da Sekar'uh' (No Wikipedia link)
[2] FOUND: 'Kempe Gowda' -> Budget: 0, Revenue: 80000000
[3] SKIPPED: 'Baby & Baby' (No Wikipedia link)
[4] SKIPPED: 'Kiccha Huccha' (No Wikipedia link)
[5] SKIPPED: 'Stockholm' (No Wikipedia link)
[6] SKIPPED: 'Bayam Ariyan' (No Wikipedia link)
[7] SKIPPED: 'Lakshya' (No Wikipedia link)
[8] EMPTY INFOBOX: 'Tourist Home' -> Budget: 0, Revenue: 0
[9] FOUND: 'Odiyan' -> Budget: 0, Revenue: 540000000
[10] SKIPPED: 'Shopner School' (No Wikipedia link)
[11] SKIPPED: 'Saakini Daakini' (No Wikipedia link)
[12] FOUND: 'Asal' -> Budget: 217000000, Revenue: 0
[13] EMPTY INFOBOX: 'Cold Case' -> Budget: 0, Revenue: 0
[14] EMPTY INFOBOX: 'Mathiya Chennai' -> Budget: 0, Revenue: 0
[15] SKIPPED: 'Darling' (No Wikipedia link)
[16] SKIPPED: 'Parole' (No Wikipedia link)
[17] FOUND: 'Bazooka' -> Budget: 0, Revenue: 2550000

## Cell 6: Split movies into those with and without finance data

This cell separates the dataset into two groups: movies that have budget or revenue data, and movies that have neither. It saves these as two separate Excel files so the successful and missing rows can be analyzed independently.

In [3]:
# --- Final Statistics ---

# Filter: Movies that have either a non-zero budget OR a non-zero revenue
df_with_finance = df[(df['budget'] > 0) | (df['revenue'] > 0)]

# Filter: Movies where BOTH budget and revenue are 0
df_no_finance = df[(df['budget'] == 0) & (df['revenue'] == 0)]

print("-" * 50)
print(f"TOTAL MOVIES PROCESSED: {len(df)}")
print(f"MOVIES WITH DATA: {len(df_with_finance)}")
print(f"MOVIES WITHOUT DATA: {len(df_no_finance)}")

# --- Saving Files ---

# Define the base directory from your file path
base_dir = os.path.dirname(file_path)

# Save the file with data
path_with_data = os.path.join(base_dir, 'movies_FOUND_finance.xlsx')
df_with_finance.to_excel(path_with_data, index=False)

# Save the file without data
path_no_data = os.path.join(base_dir, 'movies_MISSING_finance.xlsx')
df_no_finance.to_excel(path_no_data, index=False)

print("-" * 50)
print(f"Success! Created two files in: {base_dir}")
print(f"1. {os.path.basename(path_with_data)}")
print(f"2. {os.path.basename(path_no_data)}")

--------------------------------------------------
TOTAL MOVIES PROCESSED: 19286
MOVIES WITH DATA: 3324
MOVIES WITHOUT DATA: 15962
--------------------------------------------------
Success! Created two files in: C:\Users\swami\Desktop\Sem 4\MLPR\project
1. movies_FOUND_finance.xlsx
2. movies_MISSING_finance.xlsx


## Cell 7: Separate movies missing finance data but still having a Wikipedia link

This cell filters the dataset further by keeping only the movies that still have no finance data even though a Wikipedia link exists. It saves one file with movies that have finance data and another file with rows that may need manual checking.

In [4]:
# --- Final Statistics & Filtering ---

# 1. Movies that have financial data (Budget > 0 OR Revenue > 0)
df_with_finance = df[(df['budget'] > 0) | (df['revenue'] > 0)]

# 2. Movies that have NO financial data
df_no_finance = df[(df['budget'] == 0) & (df['revenue'] == 0)]

# 3. From the 'no finance' group, filter out those that don't even have a Wiki link
# This keeps only movies that HAVE a link but the scraper couldn't find data in the infobox
df_missing_but_has_link = df_no_finance[
    df_no_finance['wikipedia_link'].notna() & 
    (df_no_finance['wikipedia_link'].str.strip() != "") &
    (df_no_finance['wikipedia_link'].str.lower() != "nan")
]

print("-" * 50)
print(f"TOTAL MOVIES PROCESSED: {len(df)}")
print(f"MOVIES WITH FINANCE DATA: {len(df_with_finance)}")
print(f"MOVIES MISSING FINANCE BUT HAVE WIKI LINK: {len(df_missing_but_has_link)}")

# --- Saving Files ---

base_dir = os.path.dirname(file_path)

# File 1: Successes
path_with_data = os.path.join(base_dir, 'movies_FOUND_finance.xlsx')
df_with_finance.to_excel(path_with_data, index=False)

# File 2: Failures that have a link (The ones you might want to check manually)
path_missing_link_exists = os.path.join(base_dir, 'movies_MISSING_but_has_link.xlsx')
df_missing_but_has_link.to_excel(path_missing_link_exists, index=False)

print("-" * 50)
print(f"Success! Files saved in: {base_dir}")
print(f"1. Saved {len(df_with_finance)} rows to: {os.path.basename(path_with_data)}")
print(f"2. Saved {len(df_missing_but_has_link)} rows to: {os.path.basename(path_missing_link_exists)}")

--------------------------------------------------
TOTAL MOVIES PROCESSED: 19286
MOVIES WITH FINANCE DATA: 3324
MOVIES MISSING FINANCE BUT HAVE WIKI LINK: 6194
--------------------------------------------------
Success! Files saved in: C:\Users\swami\Desktop\Sem 4\MLPR\project
1. Saved 3324 rows to: movies_FOUND_finance.xlsx
2. Saved 6194 rows to: movies_MISSING_but_has_link.xlsx


## Cell 8: Load and clean the Indian movie dataset

This cell loads the Indian movie dataset and strips extra spaces from column names. It prepares the dataframe for the next cleaning and conversion steps.

In [3]:
import pandas as pd

# Load your file (change filename accordingly)
df = pd.read_excel("C:\\Users\\swami\\Desktop\\Sem 4\\MLPR\\project\\indian_movies_5000_unbiased.xlsx")

df.columns = df.columns.str.strip()

## Cell 9: Convert budget and revenue into Crores

This cell converts the Budget and Revenue columns into numeric values in Crores. It handles values written in Lakhs, Crores, and ranges like 15–20 Lakhs, then removes rows where either budget or revenue is zero.

In [4]:
import pandas as pd
import re

df = pd.read_excel("C:\\Users\\swami\\Desktop\\Sem 4\\MLPR\\project\\indian_movies_5000_unbiased.xlsx")

# Clean column names
df.columns = df.columns.str.strip()

# Function to convert values to Crores
def convert_money(value):
    if pd.isna(value) or value == 0:
        return 0
    
    value = str(value).replace("₹", "").strip()
    
    # Handle ranges like "15–20 Lakhs"
    if "–" in value:
        parts = value.split("–")
        nums = []
        for part in parts:
            num = float(re.findall(r"\d+\.?\d*", part)[0])
            nums.append(num)
        num = sum(nums) / len(nums)
    else:
        match = re.findall(r"\d+\.?\d*", value)
        if not match:
            return 0
        num = float(match[0])
    
    # Convert units
    if "Crore" in value:
        return num
    elif "Lakh" in value or "Lakhs" in value:
        return num / 100   # 100 Lakhs = 1 Crore
    else:
        return num

# Apply conversion
df["Budget_num"] = df["Budget"].apply(convert_money)
df["Revenue_num"] = df["Revenue"].apply(convert_money)

# Remove rows where Budget or Revenue is 0
df_cleaned = df[(df["Budget_num"] != 0) & (df["Revenue_num"] != 0)]

# Remaining rows
print("Remaining rows:", df_cleaned.shape[0])

Remaining rows: 4306


Cell 10: Adjust movie money values for inflation

This cell loads the cleaned movie dataset and adjusts the budget and revenue values to their 2025 equivalent using a fixed inflation rate. It applies the inflation formula to both budget and revenue and saves the result as inflation_adjusted_movies.xlsx.

In [22]:
import pandas as pd

# Load your Excel file
file_path = "C:\\Users\\swami\\Desktop\\Sem 4\\MLPR\\project\\Indian\\cleaned_movies.csv"  # change this to your file name
df = pd.read_csv(file_path)

# Columns:
# release_dat = release year
# Budget_num = numeric budget in Crores
# Revenue_num = numeric revenue in Crores

BASE_YEAR = 2025   # today's year
AVERAGE_INFLATION_RATE = 0.06   # 6% average yearly inflation

def adjust_for_inflation(value, release_year, base_year=BASE_YEAR, inflation_rate=AVERAGE_INFLATION_RATE):
    """
    Adjust old money value to present value using compound inflation formula:
    
    Future Value = Present Value * (1 + inflation_rate) ^ number_of_years
    """
    if pd.isna(value) or pd.isna(release_year):
        return value
    
    years = base_year - int(release_year)
    
    if years < 0:
        return value  # future year case
    
    adjusted_value = value * ((1 + inflation_rate) ** years)
    return round(adjusted_value, 2)

# Apply inflation adjustment
df["Adjusted_Budget_2025"] = df.apply(
    lambda row: adjust_for_inflation(row["Budget_num"], row["release_date"]),
    axis=1
)

df["Adjusted_Revenue_2025"] = df.apply(
    lambda row: adjust_for_inflation(row["Revenue_num"], row["release_date"]),
    axis=1
)

# Save new file
output_file = "inflation_adjusted_movies.xlsx"
df.to_excel(output_file, index=False)

print(f"Done! File saved as: {output_file}")
print(df[["title", "release_date", "Budget_num", "Adjusted_Budget_2025",
          "Revenue_num", "Adjusted_Revenue_2025"]].head())

Done! File saved as: inflation_adjusted_movies.xlsx
              title  release_date  Budget_num  Adjusted_Budget_2025  \
0         Sandhesam          1991        0.35                  2.54   
1  Shaadi Ka Laddoo          2004        4.00                 13.60   
2      Adavi Ramudu          2004        6.00                 20.40   
3           Nethaji          1996        2.50                 13.55   
4       Best Actors          2015        3.00                  5.37   

   Revenue_num  Adjusted_Revenue_2025  
0         1.75                  12.69  
1         1.80                   6.12  
2         5.00                  17.00  
3         3.00                  16.26  
4         7.50                  13.43  


## Cell 11: Split the dataset by genre availability

This cell checks which movies have missing genres and which have genre information filled in. It creates two separate files: one for movies with missing genres and one for movies with genres available.

In [23]:
import pandas as pd

# Load your Excel file
file_path = "C:\\Users\\swami\\Desktop\\Sem 4\\MLPR\\project\\inflation_adjusted_movies.xlsx"  # change if needed
df = pd.read_excel(file_path)

# Check the exact column names first
print(df.columns)

# Replace 'genres' with exact column name if different
genre_column = "genres"

# Rows where genre is missing (NaN or empty)
missing_genres_df = df[
    df[genre_column].isna() | 
    (df[genre_column].astype(str).str.strip() == "")
]

# Rows where genre is present
available_genres_df = df[
    df[genre_column].notna() & 
    (df[genre_column].astype(str).str.strip() != "")
]

# Save both files
missing_genres_df.to_excel("movies_missing_genres.xlsx", index=False)
available_genres_df.to_excel("movies_with_genres.xlsx", index=False)

print("Done!")
print(f"Missing genres rows saved: {len(missing_genres_df)}")
print(f"Available genres rows saved: {len(available_genres_df)}")

Index(['movie_id', 'title', 'language', 'release_date', 'genres', 'director',
       'actors', 'producers', 'popularity', 'vote_average', 'Budget',
       'Revenue', 'Budget.1', 'Revenue.1'],
      dtype='object')
Done!
Missing genres rows saved: 626
Available genres rows saved: 3680


In [ ]:
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# =========================
# CONFIG
# =========================
API_KEY = "903b99053bde149fce928141e82afeb0"
BASE_URL = "https://api.themoviedb.org/3"
MAX_WORKERS = 8   # ⚡ adjust (6–10 recommended)

# =========================
# LOAD DATA
# =========================
df = pd.read_excel(r"C:\Users\swami\Desktop\Sem 4\MLPR\project\movies_with_genres.xlsx")

# =========================
# HELPER
# =========================

def fetch_movie_data(index, movie_id):
    try:
        url = f"{BASE_URL}/movie/{movie_id}?api_key={API_KEY}"
        res = requests.get(url).json()

        production = None
        if res.get("production_companies"):
            production = res["production_companies"][0]["name"]

        release_date = res.get("release_date")  # full date YYYY-MM-DD

        return index, production, release_date

    except:
        return index, None, None

# =========================
# PARALLEL SCRAPING
# =========================

results = {}

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [
        executor.submit(fetch_movie_data, i, row["movie_id"])
        for i, row in df.iterrows()
    ]

    for future in tqdm(as_completed(futures), total=len(futures)):
        idx, production, release_date = future.result()
        results[idx] = (production, release_date)

# =========================
# ADD COLUMNS
# =========================

df["production_company"] = None
df["full_release_date"] = None

for idx, (prod, date) in results.items():
    df.at[idx, "production_company"] = prod
    df.at[idx, "full_release_date"] = date

# =========================
# SAVE
# =========================

df.to_excel("dataset_with_production_and_full_date.xlsx", index=False)

print("✅ DONE — production + full release date added")

100%|██████████| 4306/4306 [04:59<00:00, 14.39it/s]


✅ DONE — production + full release date added


In [9]:
import pandas as pd
import requests
from tqdm import tqdm

API_KEY = "903b99053bde149fce928141e82afeb0"
# =========================
# LOAD DATA
# =========================
df = pd.read_excel(r"C:\Users\swami\Desktop\Sem 4\MLPR\project\dataset_with_music_director.xlsx")

music_list = []

for i in tqdm(range(len(df))):
    movie_id = int(df.loc[i, "movie_id"])

    try:
        url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits?api_key={API_KEY}"
        res = requests.get(url).json()

        if "crew" not in res:
            music_list.append(None)
            continue

        music = None
        for crew in res["crew"]:
            if crew.get("job") in ["Original Music Composer", "Music"]:
                music = crew.get("name")
                break

        music_list.append(music)

    except:
        music_list.append(None)

df["music_director"] = music_list

df.to_excel("final_output.xlsx", index=False)

print("✅ Done")

100%|██████████| 4306/4306 [29:53<00:00,  2.40it/s] 


✅ Done


In [5]:
import pandas as pd

df = pd.read_excel(r"C:\Users\swami\Desktop\Sem 4\MLPR\project\movies_with_genres.xlsx")

In [7]:
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# =========================
# 🔑 LOAD DATASET (ADD YOUR FILE HERE)
# =========================
df = pd.read_excel(r"C:\Users\swami\Desktop\Sem 4\MLPR\project\movies_with_genres.xlsx")   # <-- change path if needed

print("Dataset loaded:", df.shape)

# =========================
# 🔑 CONFIG
# =========================
API_KEY = "YOUR_NEW_API_KEY"   # ⚠️ regenerate your key
BASE_URL = "https://api.themoviedb.org/3"

# =========================
# 🧠 CLASSIFICATION LOGIC
# =========================
def classify_film(languages, countries):
    indian_langs = ['hi','ta','te','ml','kn','bn','mr','pa']
    
    count = len([l for l in languages if l in indian_langs])
    
    if count >= 2:
        return "Pan-India"
    elif count == 1 and 'IN' in countries:
        return "Regional"
    else:
        return "Other"

# =========================
# 🌐 FETCH FUNCTION
# =========================
def fetch_film_type(tmdb_id):
    try:
        url = f"{BASE_URL}/movie/{tmdb_id}"
        params = {"api_key": API_KEY}
        
        response = requests.get(url, params=params, timeout=5)
        data = response.json()
        
        languages = [l['iso_639_1'] for l in data.get('spoken_languages', [])]
        countries = [c['iso_3166_1'] for c in data.get('production_countries', [])]
        
        return classify_film(languages, countries)
    
    except:
        return "Other"

# =========================
# ⚠️ CHECK COLUMN NAME
# =========================
if 'movie_id' not in df.columns:
    raise Exception("❌ 'movie_id' column not found. Rename your TMDB ID column to 'movie_id'")

# =========================
# 🚀 PARALLEL EXECUTION
# =========================
film_types = [None] * len(df)

with ThreadPoolExecutor(max_workers=15) as executor:   # safer than 20
    futures = {
        executor.submit(fetch_film_type, tmdb_id): idx 
        for idx, tmdb_id in enumerate(df['movie_id'])
    }
    
    for i, future in enumerate(as_completed(futures)):
        idx = futures[future]
        film_types[idx] = future.result()
        
        # progress
        if i % 100 == 0:
            print(f"Processed {i} movies...")

# Add column
df['film_type'] = film_types

# =========================
# 🔢 ENCODE
# =========================
df = pd.get_dummies(df, columns=['film_type'], drop_first=True)

# =========================
# 💾 SAVE FILE
# =========================
output_path = "movies_with_type"
df.to_csv(output_path, index=False)

# =========================
# ✅ DONE
# =========================
print("✅ Done!")
print("📁 File saved as:", output_path)
print(df[['film_type_Pan-India', 'film_type_Regional']].head())

Dataset loaded: (4306, 12)
Processed 0 movies...
Processed 100 movies...
Processed 200 movies...
Processed 300 movies...
Processed 400 movies...
Processed 500 movies...
Processed 600 movies...
Processed 700 movies...
Processed 800 movies...
Processed 900 movies...
Processed 1000 movies...
Processed 1100 movies...
Processed 1200 movies...
Processed 1300 movies...
Processed 1400 movies...
Processed 1500 movies...
Processed 1600 movies...
Processed 1700 movies...
Processed 1800 movies...
Processed 1900 movies...
Processed 2000 movies...
Processed 2100 movies...
Processed 2200 movies...
Processed 2300 movies...
Processed 2400 movies...
Processed 2500 movies...
Processed 2600 movies...
Processed 2700 movies...
Processed 2800 movies...
Processed 2900 movies...
Processed 3000 movies...
Processed 3100 movies...
Processed 3200 movies...
Processed 3300 movies...
Processed 3400 movies...
Processed 3500 movies...
Processed 3600 movies...
Processed 3700 movies...
Processed 3800 movies...
Processed 

KeyError: "None of [Index(['film_type_Pan-India', 'film_type_Regional'], dtype='object')] are in the [columns]"

In [3]:
import pandas as pd

# Load your dataset
df = pd.read_excel(r"C:\Users\swami\Desktop\Sem 4\MLPR\project\dataset_with_music_director.xlsx")

# Make sure column name is correct (check spelling!)
print(df.columns)

# Clean column (important)
df['music_director'] = df['music_director'].astype(str).str.strip()

# Split dataset
with_music = df[df['music_director'].notna() & (df['music_director'] != '') & (df['music_director'] != 'nan')]
without_music = df[~(df['music_director'].notna() & (df['music_director'] != '') & (df['music_director'] != 'nan'))]

# Save both files
with_music.to_excel("with_music.xlsx", index=False)
without_music.to_excel("without_music.xlsx", index=False)

print("✅ Files created successfully!")
print("With music:", with_music.shape)
print("Without music:", without_music.shape)

Index(['movie_id', 'title', 'language', 'release_date', 'genres', 'director',
       'actors', 'producers', 'popularity', 'vote_average', 'Budget',
       'Revenue', 'Budget.1', 'Revenue.1', 'music_director',
       'full_release_date'],
      dtype='object')
✅ Files created successfully!
With music: (2952, 16)
Without music: (1354, 16)


In [1]:
import pandas as pd

df = pd.read_excel(r"C:\Users\swami\AppData\Local\Packages\5319275A.WhatsAppDesktop_cv1g1gvanyjgm\LocalState\sessions\B6CEBA2B3316D8E7AB945F7B07CC6C1C63867EA4\transfers\2026-18\unbiased_fast_dataset.xlsx")   # or your file name

print(df.head())
print(df.columns)

   movie_id                                     title release_date  \
0    157336                              Interstellar   2014-11-05   
1       671  Harry Potter and the Philosopher's Stone   2001-11-16   
2       680                              Pulp Fiction   1994-09-10   
3    118340                   Guardians of the Galaxy   2014-07-30   
4       597                                   Titanic   1997-12-18   

                               genres           director  \
0   Adventure, Drama, Science Fiction  Christopher Nolan   
1                  Adventure, Fantasy     Chris Columbus   
2             Thriller, Crime, Comedy  Quentin Tarantino   
3  Action, Science Fiction, Adventure         James Gunn   
4                      Drama, Romance      James Cameron   

                                              actors  \
0  Matthew McConaughey, Anne Hathaway, Michael Ca...   
1  Daniel Radcliffe, Rupert Grint, Emma Watson, R...   
2  John Travolta, Samuel L. Jackson, Uma Thurman,.

In [2]:
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# =========================
# 🔑 CONFIG
# =========================
API_KEY = "903b99053bde149fce928141e82afeb0"   # ⚠️ regenerate your key
BASE_URL = "https://api.themoviedb.org/3"

# =========================
# 🌐 FETCH FUNCTION
# =========================
def fetch_production_company(tmdb_id):
    try:
        url = f"{BASE_URL}/movie/{tmdb_id}"
        params = {"api_key": API_KEY}
        
        response = requests.get(url, params=params, timeout=5)
        data = response.json()
        
        companies = data.get('production_companies', [])
        
        # Extract company names
        company_names = [c['name'] for c in companies]
        
        return company_names if company_names else []
    
    except:
        return []

# =========================
# 🚀 PARALLEL EXECUTION
# =========================

production_companies = [None] * len(df)

with ThreadPoolExecutor(max_workers=20) as executor:
    futures = {
        executor.submit(fetch_production_company, tmdb_id): idx
        for idx, tmdb_id in enumerate(df['movie_id'])
    }
    
    for i, future in enumerate(as_completed(futures)):
        idx = futures[future]
        production_companies[idx] = future.result()
        
        if i % 100 == 0:
            print(f"Processed {i} movies...")

# Add column
df['production_companies'] = production_companies

print("✅ Production companies added!")
print(df[['production_companies']].head())

Processed 0 movies...
Processed 100 movies...
Processed 200 movies...
Processed 300 movies...
Processed 400 movies...
Processed 500 movies...
Processed 600 movies...
Processed 700 movies...
Processed 800 movies...
Processed 900 movies...
Processed 1000 movies...
Processed 1100 movies...
Processed 1200 movies...
Processed 1300 movies...
Processed 1400 movies...
Processed 1500 movies...
Processed 1600 movies...
Processed 1700 movies...
Processed 1800 movies...
Processed 1900 movies...
Processed 2000 movies...
Processed 2100 movies...
Processed 2200 movies...
Processed 2300 movies...
Processed 2400 movies...
Processed 2500 movies...
Processed 2600 movies...
Processed 2700 movies...
Processed 2800 movies...
Processed 2900 movies...
Processed 3000 movies...
Processed 3100 movies...
Processed 3200 movies...
Processed 3300 movies...
Processed 3400 movies...
Processed 3500 movies...
Processed 3600 movies...
Processed 3700 movies...
Processed 3800 movies...
Processed 3900 movies...
Processed 40

In [3]:
df.to_excel("final_with_production_company.xlsx", index=False)

In [10]:
import pandas as pd
import ast

# Load dataset
df = pd.read_excel(r"C:\Users\swami\Desktop\Sem 4\MLPR\project\Hollywood\final_with_production_company.xlsx")   # or your file

# =========================
# 🔧 FIX COLUMN (convert string → list if needed)
# =========================
def convert_to_list(x):
    try:
        if isinstance(x, list):
            return x
        return ast.literal_eval(x)
    except:
        return []

df['production_companies'] = df['production_companies'].apply(convert_to_list)

# =========================
# 🔍 CREATE MASK
# =========================
mask = df['production_companies'].apply(lambda x: isinstance(x, list) and len(x) > 0)

# =========================
# ✂️ SPLIT DATA
# =========================
with_pro = df[mask]
without_pro = df[~mask]

# =========================
# 💾 SAVE FILES
# =========================
with_pro.to_csv("with_production_company.csv", index=False)
without_pro.to_csv("without_production_company.csv", index=False)

print("✅ Done!")
print("With production:", with_pro.shape)
print("Without production:", without_pro.shape)

✅ Done!
With production: (3527, 13)
Without production: (1362, 13)


In [1]:
import pandas as pd
import requests
from datetime import datetime
from tqdm import tqdm
import time

# =========================
# 1) LOAD DATA
# =========================
df = pd.read_csv(r"C:\Users\swami\Desktop\Sem 4\MLPR\project\Hollywood\with_prod_comp_Hollywood.csv")

# =========================
# 2) TMDb CONFIG
# =========================
API_KEY = "903b99053bde149fce928141e82afeb0"   # ⚠️ regenerate your key
BASE_URL = "https://api.themoviedb.org/3"

session = requests.Session()

# =========================
# 3) HELPERS
# =========================
def clean_list(text):
    if pd.isna(text):
        return []
    return [x.strip() for x in str(text).split(",") if x.strip()]

def calculate_age(birthday):
    if not birthday:
        return None
    try:
        b = datetime.strptime(birthday, "%Y-%m-%d")
        today = datetime.today()
        return today.year - b.year - ((today.month, today.day) < (b.month, b.day))
    except:
        return None

def get_person_age(name):
    try:
        search = session.get(
            f"{BASE_URL}/search/person",
            params={"api_key": API_KEY, "query": name}
        ).json()

        if not search["results"]:
            return None

        person_id = search["results"][0]["id"]

        details = session.get(
            f"{BASE_URL}/person/{person_id}",
            params={"api_key": API_KEY}
        ).json()

        return calculate_age(details.get("birthday"))

    except:
        return None

# =========================
# 4) BUILD UNIQUE NAME SET  🔥 (THIS IS THE SPEED TRICK)
# =========================
all_people = set()

for _, row in df.iterrows():
    # top actor
    actors = clean_list(row["actors"])
    if actors:
        all_people.add(actors[0])

    # director
    if pd.notna(row["director"]):
        all_people.add(row["director"])

    # producers
    for p in clean_list(row["producers"]):
        all_people.add(p)

print(f"Unique people to fetch: {len(all_people)}")

# =========================
# 5) FETCH AGES ONCE ONLY
# =========================
age_map = {}

for person in tqdm(all_people):
    age_map[person] = get_person_age(person)
    time.sleep(0.15)   # small delay to avoid rate limit

# =========================
# 6) MAP BACK TO DATASET
# =========================
top_actor_list = []
top_actor_age_list = []
director_age_list = []
producer_age_list = []

for _, row in df.iterrows():
    
    # Top actor
    actors = clean_list(row["actors"])
    top_actor = actors[0] if actors else None
    
    top_actor_list.append(top_actor)
    top_actor_age_list.append(age_map.get(top_actor))
    
    # Director
    director = row["director"]
    director_age_list.append(age_map.get(director))
    
    # Producers
    producers = clean_list(row["producers"])
    ages = [str(age_map.get(p)) for p in producers]
    producer_age_list.append(" | ".join(ages))

# =========================
# 7) ADD COLUMNS
# =========================
df["top_actor"] = top_actor_list
df["top_actor_age"] = top_actor_age_list
df["director_age"] = director_age_list
df["producers_age"] = producer_age_list

# =========================
# 8) SAVE
# =========================
df.to_csv("dataset_with_ages.csv", index=False)

print("Saved: fast_final_dataset.csv")

# =========================
# 9) DOWNLOAD (COLAB)
# =========================
try:
    from google.colab import files
    files.download("fast_final_dataset.csv")
except:
    pass

Unique people to fetch: 6219


100%|██████████| 6219/6219 [1:37:20<00:00,  1.06it/s]


Saved: fast_final_dataset.csv


In [2]:
pip install pandas cpi

   ---------------------------------------- 0.0/18.1 MB ? eta -:--:--
   -- ------------------------------------- 1.0/18.1 MB 49.2 MB/s eta 0:00:01
   -- ------------------------------------- 1.3/18.1 MB 2.9 MB/s eta 0:00:06
   ---- ----------------------------------- 2.1/18.1 MB 4.7 MB/s eta 0:00:04
   ---- ----------------------------------- 2.1/18.1 MB 4.7 MB/s eta 0:00:04
   ---- ----------------------------------- 2.1/18.1 MB 4.7 MB/s eta 0:00:04
   ---- ----------------------------------- 2.1/18.1 MB 4.7 MB/s eta 0:00:04
   ----- ---------------------------------- 2.4/18.1 MB 1.5 MB/s eta 0:00:11
   ------ --------------------------------- 3.1/18.1 MB 2.0 MB/s eta 0:00:08
   --------- ------------------------------ 4.2/18.1 MB 2.2 MB/s eta 0:00:07
   --------- ------------------------------ 4.2/18.1 MB 2.2 MB/s eta 0:00:07
   ----------- ---------------------------- 5.2/18.1 MB 2.3 MB/s eta 0:00:06
   ----------- ---------------------------- 5.2/18.1 MB 2.3 MB/s eta 0:00:06
   --


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import pandas as pd
import cpi

# 1. Load your dataset
df = pd.read_excel("C:\\Users\\swami\\Desktop\\Sem 4\\MLPR\\project\\Hollywood\\inflation_adjusted_movies.xlsx")

# Update CPI once
try:
    cpi.update()
except:
    pass

# 2. FIXED: Detect valid target year by checking a real historical conversion
target_year = 2024  # Safe baseline fallback
for y in [2026, 2025, 2024, 2023]:
    try:
        # This forces the library to actually look up 'y' in the database
        cpi.inflate(100, 2000, to=y)
        target_year = y
        break
    except:
        continue

print(f"Using verified target year: {target_year}")

# Clean the year column up front to prevent float/int mismatch issues
df["year"] = pd.to_numeric(df["year"], errors='coerce').fillna(target_year).astype(int)

# 3. PRECOMPUTE multipliers
years = df["year"].unique()
inflation_map = {}

for y in years:
    try:
        if y < 1913 or y >= target_year:
            inflation_map[y] = 1.0
        else:
            # This will now work flawlessly because target_year is verified to exist
            inflation_map[y] = cpi.inflate(1, y, to=target_year)
    except Exception as e:
        # If an individual year fails, fallback to 1.0
        print(f"Warning: Could not calculate inflation for year {y}. Using 1.0 fallback.")
        inflation_map[y] = 1.0

# 4. Vectorized multiplication (FAST)
df["inflation_multiplier"] = df["year"].map(inflation_map).fillna(1.0)

df[f"budget_{target_year}_adjusted"] = df["budget"] * df["inflation_multiplier"]
df[f"revenue_{target_year}_adjusted"] = df["revenue"] * df["inflation_multiplier"]

# Clean up the multiplier feature before saving
df.drop(columns=["inflation_multiplier"], inplace=True)

# 5. Save the final file
output_path = f"movies_adjusted_to_{target_year}.csv"
df.to_csv(output_path, index=False)

print(f"Done! Values successfully adjusted to {target_year} and saved.")

c:\Users\swami\AppData\Local\Programs\Python\Python312\Lib\site-packages\cpi\download.py:175: DtypeWarning: Columns (3,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.StringIO(response.text), sep="\t")
c:\Users\swami\AppData\Local\Programs\Python\Python312\Lib\site-packages\cpi\download.py:175: DtypeWarning: Columns (3,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.StringIO(response.text), sep="\t")
c:\Users\swami\AppData\Local\Programs\Python\Python312\Lib\site-packages\cpi\download.py:175: DtypeWarning: Columns (3,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.StringIO(response.text), sep="\t")
c:\Users\swami\AppData\Local\Programs\Python\Python312\Lib\site-packages\cpi\download.py:175: DtypeWarning: Columns (3,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.StringIO(response.text)

Using verified target year: 2025
Done! Values successfully adjusted to 2025 and saved.


In [15]:
import pandas as pd
import cpi

# 1. Load the new production companies dataset
input_file = r"C:\Users\swami\Desktop\Sem 4\MLPR\project\Hollywood\with_prod_comp_Hollywood.csv"
df = pd.read_csv(input_file)

print("Updating internal CPI data index...")
try:
    cpi.update()
except Exception:
    pass  # Automatically uses local fallback if offline

# 2. Automatically detect maximum verified target year
target_year = 2024  
for y in [2026, 2025, 2024, 2023]:
    try:
        cpi.inflate(100, 2000, to=y)
        target_year = y
        break
    except Exception:
        continue

print(f"Verified conversion anchor year: {target_year}")

# 3. NEW STEP: Extract year directly from the 'release_date' feature string
df["year_clean"] = pd.to_datetime(df["release_date"], errors='coerce').dt.year
# Fill missing dates if any exist with the target year baseline, then cast to integer
df["year_clean"] = df["year_clean"].fillna(target_year).astype(int)

# 4. PRECOMPUTE multipliers for speed (1 lookup per unique year)
print("Calculating historical inflation rates...")
unique_years = df["year_clean"].unique()
inflation_map = {}

for y in unique_years:
    try:
        if y < 1913 or y >= target_year:
            inflation_map[y] = 1.0
        else:
            inflation_map[y] = cpi.inflate(1, y, to=target_year)
    except Exception:
        inflation_map[y] = 1.0

# 5. Apply lightning-fast vectorized calculation across all rows
df["multiplier"] = df["year_clean"].map(inflation_map).fillna(1.0)

df[f"budget_{target_year}_adjusted"] = df["budget"] * df["multiplier"]
df[f"revenue_{target_year}_adjusted"] = df["revenue"] * df["multiplier"]

# 6. Housekeeping: Drop temporary multiplier column
# (We keep 'year_clean' as it's a valuable structured continuous feature for ML training)
df.drop(columns=["multiplier"], inplace=True)
df.rename(columns={"year_clean": "year"}, inplace=True)

# 7. Save out your immaculate new machine learning dataset
output_file = f"with_prod_comp_Hollywood_adjusted_{target_year}.csv"
df.to_csv(output_file, index=False)

print(f"Success! Adjusted file saved to disk as: '{output_file}'")

Updating internal CPI data index...


c:\Users\swami\AppData\Local\Programs\Python\Python312\Lib\site-packages\cpi\download.py:175: DtypeWarning: Columns (3,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.StringIO(response.text), sep="\t")
c:\Users\swami\AppData\Local\Programs\Python\Python312\Lib\site-packages\cpi\download.py:175: DtypeWarning: Columns (3,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.StringIO(response.text), sep="\t")
c:\Users\swami\AppData\Local\Programs\Python\Python312\Lib\site-packages\cpi\download.py:175: DtypeWarning: Columns (3,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.StringIO(response.text), sep="\t")
c:\Users\swami\AppData\Local\Programs\Python\Python312\Lib\site-packages\cpi\download.py:175: DtypeWarning: Columns (3,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.StringIO(response.text)

Verified conversion anchor year: 2025
Calculating historical inflation rates...
Success! Adjusted file saved to disk as: 'with_prod_comp_Hollywood_adjusted_2025.csv'
